<a href="https://colab.research.google.com/github/engcarto-Paulo/WebSIg-Desenvolvimento/blob/main/cod-desvolvimento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. INSTALAÇÃO E IMPORTAÇÃO DE BIBLIOTECAS
# ==========================================
!pip install rasterio geopandas folium

import folium
from folium import plugins
from folium.plugins import Draw, MeasureControl
import rasterio
from rasterio.plot import reshape_as_image
from rasterio.warp import calculate_default_transform, reproject, Resampling
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
from branca.element import Template, MacroElement
from PIL import Image

# Conectar ao Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ==========================================
# 2. REPROJEÇÃO DO RASTER COM REDUÇÃO DE TAMANHO (DOWNSAMPLING = 10)
# ==========================================
orto_original = '/content/drive/My Drive/dados/ORTOMOSAICO.tif'
orto_wgs84 = '/content/orto_wgs84.tif'
dst_crs = 'EPSG:4326'

print("Reprojetando e otimizando o ortomosaico para Web (Fator 10)...")

with rasterio.open(orto_original) as src:
    nodata_val = src.nodata

    # Reduz as dimensões por 10 para o arquivo final ficar com menos de 25MB
    fator_reducao = 10
    nova_largura = int(src.width / fator_reducao)
    nova_altura = int(src.height / fator_reducao)

    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, nova_largura, nova_altura, *src.bounds
    )

    kwargs = src.meta.copy()
    kwargs.update({
        'crs': dst_crs,
        'transform': transform,
        'width': width,
        'height': height
    })

    with rasterio.open(orto_wgs84, 'w', **kwargs) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.bilinear
            )

print("Reprojeção e compressão concluídas!")

# ==========================================
# 3. CONVERSÃO PARA RGBA COM COMPRESSÃO MÁXIMA (PIL)
# ==========================================
with rasterio.open(orto_wgs84) as src:
    img = src.read([1, 2, 3])
    bounds = src.bounds
    nodata_val = src.nodata

if nodata_val is None:
    nodata_val = 0

img = reshape_as_image(img)

is_nodata = (img[:, :, 0] == nodata_val) & (img[:, :, 1] == nodata_val) & (img[:, :, 2] == nodata_val)
is_white_border = (img[:, :, 0] >= 250) & (img[:, :, 1] >= 250) & (img[:, :, 2] >= 250)

alpha = np.ones((img.shape[0], img.shape[1]), dtype=np.uint8) * 255
alpha[is_nodata | is_white_border] = 0

img_norm = img.astype(float)
for i in range(3):
    min_v = np.percentile(img_norm[:,:,i], 2)
    max_v = np.percentile(img_norm[:,:,i], 98)
    img_norm[:,:,i] = np.clip((img_norm[:,:,i] - min_v) / (max_v - min_v + 1e-5) * 255, 0, 255)
img_norm = img_norm.astype(np.uint8)

img_rgba = np.dstack((img_norm, alpha))

# Salvamento otimizado com Pillow para reduzir drasticamente o tamanho do HTML
img_pil = Image.fromarray(img_rgba)
img_pil.save('/content/ortomosaico_transparente.png', 'PNG', optimize=True, compress_level=9)
print("Imagem com fundo transparente e ultra-compactada gerada com sucesso!")

# ==========================================
# 4. CRIAÇÃO DO MAPA INTERATIVO (FOLIUM + SATÉLITES)
# ==========================================
centro_lat = (bounds.bottom + bounds.top) / 2
centro_lon = (bounds.left + bounds.right) / 2

m = folium.Map(
    location=[centro_lat, centro_lon],
    tiles='OpenStreetMap',
    zoom_start=16
)

# Adicionamos o Google Satélite usando subdomínios estáveis
folium.TileLayer(
    tiles='https://{s}.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
    subdomains=['mt0', 'mt1', 'mt2', 'mt3'],
    attr='Google',
    name='Satélite (Google)',
    overlay=False
).add_to(m)

# Adicionamos o Satélite Esri
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri | DigitalGlobe, GeoEye, i-cubed, USDA, USGS, AEX, Getmapping, Aerogrid, IGN, IGP, swisstopo, and the GIS User Community',
    name='Satélite (Esri)',
    overlay=False
).add_to(m)

# Mapas de fundo CartoDB
folium.TileLayer("CartoDB positron", name="Mapa Claro (CartoDB)", overlay=False).add_to(m)
folium.TileLayer("CartoDB dark_matter", name="Mapa Escuro (CartoDB)", overlay=False).add_to(m)

# Adicionando o seu Ortomosaico Sem Bordas
folium.raster_layers.ImageOverlay(
    image='/content/ortomosaico_transparente.png',
    bounds=[[bounds.bottom, bounds.left],
            [bounds.top, bounds.right]],
    opacity=1.0,
    name="Ortomosaico Limpo",
    overlay=True
).add_to(m)

# 🌟 TRANCAR O ZOOM NA ÁREA CERTA (Garante abertura correta sem precisar caçar o mapa)
m.fit_bounds([[bounds.bottom, bounds.left], [bounds.top, bounds.right]])

# ==========================================
# 5. CARREGAMENTO DOS DADOS E CONFIGURAÇÃO DOS POP-UPS
# ==========================================

# --- LOTES ---
caminho_lotes = '/content/drive/My Drive/dados/lotes.geojson'
gdf_lotes = gpd.read_file(caminho_lotes).to_crs(epsg=4326)

colunas_lotes = list(gdf_lotes.columns)
if 'geometry' in colunas_lotes: colunas_lotes.remove('geometry')

popup_lotes = folium.features.GeoJsonPopup(
    fields=colunas_lotes,
    aliases=colunas_lotes,
    labels=True,
    style="font-family: Arial; font-size: 12px; background-color: #f9f9f9;"
)

folium.GeoJson(
    gdf_lotes,
    name="Lotes",
    style_function=lambda x: {
        'color': '#E6C653',      # Amarelo pastel menos saturado
        'fillColor': 'rgba(0,0,0,0)',
        'fillOpacity': 0,
        'weight': 2
    },
    popup=popup_lotes
).add_to(m)

# --- EDIFICAÇÕES ---
caminho_edificacoes = '/content/drive/My Drive/dados/edificacoes.geojson'
gdf_edificacoes = gpd.read_file(caminho_edificacoes).to_crs(epsg=4326)

colunas_edificacoes = list(gdf_edificacoes.columns)
if 'geometry' in colunas_edificacoes: colunas_edificacoes.remove('geometry')

popup_edificacoes = folium.features.GeoJsonPopup(
    fields=colunas_edificacoes,
    aliases=colunas_edificacoes,
    labels=True,
    style="font-family: Arial; font-size: 12px; background-color: #f9f9f9;"
)

folium.GeoJson(
    gdf_edificacoes,
    name="Edificações",
    style_function=lambda x: {
        'fillColor': '#CD5C5C',   # Vermelho Terracota
        'color': '#8B0000',       # Borda Vermelho Escuro
        'fillOpacity': 0.6,
        'weight': 1.5
    },
    popup=popup_edificacoes
).add_to(m)

# --- EIXO DE RUA ---
caminho_eixos = '/content/drive/My Drive/dados/eixo_de_rua.geojson'
gdf_eixos = gpd.read_file(caminho_eixos).to_crs(epsg=4326)

colunas_eixos = list(gdf_eixos.columns)
if 'geometry' in colunas_eixos: colunas_eixos.remove('geometry')

popup_eixos = folium.features.GeoJsonPopup(
    fields=colunas_eixos,
    aliases=colunas_eixos,
    labels=True,
    style="font-family: Arial; font-size: 12px; background-color: #f9f9f9;"
)

folium.GeoJson(
    gdf_eixos,
    name="Eixo de Rua",
    style_function=lambda x: {
        'color': '#2F4F4F',
        'weight': 2.5
    },
    popup=popup_eixos
).add_to(m)

# --- ÁREA VERDE ---
caminho_areas_verdes = '/content/drive/My Drive/dados/area_verde.geojson'
gdf_areas_verdes = gpd.read_file(caminho_areas_verdes).to_crs(epsg=4326)

colunas_verdes = list(gdf_areas_verdes.columns)
if 'geometry' in colunas_verdes: colunas_verdes.remove('geometry')

popup_verdes = folium.features.GeoJsonPopup(
    fields=colunas_verdes,
    aliases=colunas_verdes,
    labels=True,
    style="font-family: Arial; font-size: 12px; background-color: #f9f9f9;"
)

folium.GeoJson(
    gdf_areas_verdes,
    name="Área Verde",
    style_function=lambda x: {
        'fillColor': '#228B22',
        'color': '#006400',
        'fillOpacity': 0.5,
        'weight': 1.5
    },
    popup=popup_verdes
).add_to(m)

# ==========================================
# 6. ADICIONANDO LEGENDA PARA AS CAMADAS
# ==========================================
template_legenda = """
{% macro html(this, kwargs) %}
<div id='maplegend' class='maplegend'
    style='position: fixed; z-index:9999; border: 1px solid #e0e0e0; background-color: rgba(255, 255, 255, 0.98);
     border-radius: 8px; padding: 12px; font-size: 13px; right: 20px; bottom: 20px; font-family: "Segoe UI", Arial, sans-serif;
     box-shadow: 0 4px 12px rgba(0,0,0,0.1); min-width: 160px; color: #333;'>

  <div class='legend-scale'>
    <ul class='legend-labels' style='margin: 0; padding: 0; list-style: none;'>

      <li style='margin-bottom: 8px; display: flex; align-items: center;'>
        <span style='display: inline-block; width: 24px; height: 4px; background: #E6C653; border-radius: 2px; margin-right: 10px;'></span>
        <span style='font-weight: 500;'>Lotes (Limite)</span>
      </li>

      <li style='margin-bottom: 8px; display: flex; align-items: center;'>
        <span style='display: inline-block; width: 24px; height: 3px; background: #4A5555; border-radius: 1px; margin-right: 10px;'></span>
        <span style='font-weight: 500;'>Eixo de Rua</span>
      </li>

      <li style='margin-bottom: 8px; display: flex; align-items: center;'>
        <span style='display: inline-block; width: 20px; height: 14px; background: rgba(205, 92, 92, 0.7); border: 1.5px solid #8B0000; border-radius: 3px; margin-right: 10px;'></span>
        <span style='font-weight: 500;'>Edificações</span>
      </li>

      <li style='margin-bottom: 8px; display: flex; align-items: center;'>
        <span style='display: inline-block; width: 20px; height: 14px; background: rgba(34, 139, 34, 0.5); border: 1.5px solid #006400; border-radius: 3px; margin-right: 10px;'></span>
        <span style='font-weight: 500;'>Área Verde</span>
      </li>

      <li style='display: flex; align-items: center;'>
        <span style='display: inline-block; width: 20px; height: 14px; margin-right: 10px; border: 1px solid #777; border-radius: 2px;
                     background-image: linear-gradient(45deg, #ccc 25%, transparent 25%), linear-gradient(-45deg, #ccc 25%, transparent 25%), linear-gradient(45deg, transparent 75%, #ccc 75%), linear-gradient(-45deg, transparent 75%, #ccc 75%);
                     background-size: 6px 6px; background-position: 0 0, 0 3px, 3px -3px, -3px 0px;'></span>
        <span style='font-weight: 500;'>Ortomosaico</span>
      </li>

    </ul>
  </div>
</div>
{% endmacro %}
"""

macro = MacroElement()
macro._template = Template(template_legenda)
m.get_root().add_child(macro)

# ==========================================
# 7. ADICIONANDO FERRAMENTAS DE MEDIÇÃO E DESENHO
# ==========================================
MeasureControl(
    position='topleft',
    primary_length_unit='meters',
    secondary_length_unit='kilometers',
    primary_area_unit='sqmeters',
    secondary_area_unit='hectares'
).add_to(m)

Draw(
    export=True,
    filename='dados_coletados.geojson',
    position='topleft',
    draw_options={
        'polyline': {'shapeOptions': {'color': '#ff0000', 'weight': 3}},
        'polygon': {'showArea': True, 'metric': True, 'shapeOptions': {'color': '#138808'}},
        'rectangle': {'shapeOptions': {'color': '#ff9933'}},
        'marker': True,
        'circle': False,
        'circlemarker': False
    },
    edit_options={
        'edit': True,
        'remove': True
    }
).add_to(m)

# Adiciona o seletor de camadas
folium.LayerControl().add_to(m)

# 1. Salva o arquivo HTML localmente no Colab com o nome adequado
caminho_local_colab = '/content/WebSig-2.html'
m.save(caminho_local_colab)
print("Mapa compactado e gerado com sucesso!")

# 2. Executa a transferência automática direta para o seu computador
from google.colab import files
files.download(caminho_local_colab)

# 3. Exibe o mapa no notebook
m